# Web Scraping: Highest-Grossing Films from Wikipedia

## Overview
This notebook scrapes data about the highest-grossing films from Wikipedia, 
extracts detailed information from each film's page, and stores the results 
in CSV, SQLite, and JSON formats.

## Data Pipeline
1. **Scraping** - Extract film list from Wikipedia
2. **Parsing** - Get additional details (director, country, production companies)
3. **Storage** - Save to CSV → SQLite → JSON

## Dependencies
- requests
- beautifulsoup4
- pandas
- sqlite3

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
import random
import sqlite3
import json

# Web Scraping: Highest-Grossing Films from Wikipedia

## Objective
Collect data about the highest-grossing films in cinema history from Wikipedia and prepare it for visualization in a 3D film galaxy.

## Data Source
- URL: https://en.wikipedia.org/wiki/List_of_highest-grossing_films
- Target table: wikitable with film rankings

## Data Fields Collected
1. Film title
2. Release year
3. Box office earnings (in USD)
4. Director(s)
5. Country of origin
6. Production company/studio

## Libraries Used
- `requests` - HTTP requests
- `BeautifulSoup` - HTML parsing
- `re` - text cleaning
- `time` and `random` - delays between requests

## Cleaning Functions
- `clean_text()` - removes reference markers like [1], [note 2] and extra whitespace
- `clean_money()` - converts box office string to float number
- `extract_cell_text()` - extracts clean text from table cell

## Implementation Features
- Random delays between requests (0.5-1.2 seconds)
- Retry logic on failures (up to 3 attempts)
- Infobox parsing for each film's detailed information

In [2]:

# URL of the Wikipedia page for Highest-Grossing Films
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"

# Headers to mimic a real browser request (prevents blocking)
headers = {'User-Agent': 'Mozilla/5.0'}


# ---------- Function: Safe HTTP Request ----------
def safe_request(url, retries=3):
    """
    Sends an HTTP GET request with retries and returns a valid response.
    - url: the target URL
    - retries: number of attempts
    Returns: requests.Response object or None if all attempts fail
    """
    for _ in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            # Check that the page is returned successfully and is not empty
            if response.status_code == 200 and len(response.text) > 1000:
                return response
        except:
            pass
        # Random delay between retries to avoid server throttling
        time.sleep(random.uniform(0.5, 1.5))
    return None


# ---------- Function: Clean Text ----------
def clean_text(text):
    """
    Cleans a string by:
    - Removing reference markers like [1], [note 2], etc.
    - Splitting camelCase words with a comma (optional for director names)
    - Removing currency symbols
    - Removing extra whitespace
    Returns: cleaned string
    """
    # Remove square-bracket references
    text = re.sub(r'\[\s*[^\]]*?\]', '', text)
    
    # Add comma between lowercase-uppercase joins (e.g., "JonWatts" → "Jon, Watts")
    text = re.sub(r'([a-z])([A-Z])', r'\1, \2', text)
    
    # Remove currency symbols
    text = re.sub(r'[\$€£]', '', text)
    
    # Remove multiple spaces
    return re.sub(r'\s+', ' ', text).strip()


# ---------- Function: Clean Money ----------
def clean_money(text):
    """
    Converts a box office string to a float value.
    - Removes commas and symbols
    - Finds numeric substrings
    - Returns the largest number as float
    Returns: float or None if parsing fails
    """
    text = clean_text(text)
    text = text.replace(',', '')

    # Find all numeric parts
    numbers = re.findall(r'\d+', text)
    if not numbers:
        return None

    # Pick the longest number as the likely box office
    return float(max(numbers, key=len))


# ---------- Function: Extract Cell Text ----------
def extract_cell_text(cell):
    """
    Extracts cleaned text from a table cell (td or th):
    - Removes <sup> tags (footnotes)
    - Removes duplicates
    - Joins multiple parts with ';'
    Returns: string or None
    """
    if not cell:
        return None

    # Remove <sup> tags
    for sup in cell.find_all('sup'):
        sup.decompose()

    # Extract text strings
    parts = list(cell.stripped_strings)
    parts = [clean_text(p) for p in parts if clean_text(p)]

    # Remove any remaining brackets or digits
    parts = [p for p in parts if not re.fullmatch(r'[\[\]\d]+', p)]

    # Remove duplicates while preserving order
    parts = list(dict.fromkeys(parts))

    return '; '.join(parts) if parts else None


# ---------- Function: Parse Individual Film Page ----------
def parse_film_page(film_url):
    """
    Visits the Wikipedia page of an individual film to extract:
    - Director(s)
    - Country of origin
    - Production company names
    Returns: director, country, production (all strings)
    """
    director, country, production = None, None, None

    response = safe_request(film_url)
    if not response:
        return None, None, None

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find the infobox on the right side of the page
    infobox = soup.find('table', class_=lambda x: x and 'infobox' in x)
    if not infobox:
        return None, None, None

    # Iterate over all rows in the infobox
    for row in infobox.find_all('tr'):
        # First try th as header, then check for td with 'infobox-label'
        header = row.find('th')
        if not header:
            header = row.find('td', class_=lambda x: x and 'infobox-label' in x)
        
        if not header:
            continue
        
        # Look for the corresponding data cell
        data = row.find('td')
        if not data:
            data = header.find_next_sibling('td')
        if not data:
            next_row = row.find_next_sibling('tr')
            if next_row:
                data = next_row.find('td')
        if not data:
            continue
        
        header_text = header.get_text(separator=' ', strip=True).lower()
        
        # ---------- Extract Director ----------
        if any(k in header_text for k in ['directed by', 'director', 'directors', 'directed']):
            cell_text = extract_cell_text(data)
            if cell_text:
                director = cell_text
                
        # ---------- Extract Country ----------
        if any(k in header_text for k in ['country', 'countries', 'origin', 'nationality']):
            cell_text = extract_cell_text(data)
            if cell_text:
                country = cell_text
                
        # ---------- Extract Production Companies ----------
        if any(k in header_text for k in ['production', 'company', 'studio']):
            cell_text = extract_cell_text(data)
            if cell_text:
                production = cell_text

    # ---------- Fallback using first 3 paragraphs ----------
    if not director or not country:
        paragraphs = soup.find_all('p')
        text = ' '.join(p.get_text() for p in paragraphs[:3])

        if not director:
            match = re.search(r'directed by ([A-Z][a-zA-Z\s,]+)', text)
            if match:
                director = match.group(1)

        if not country:
            match = re.search(r'([A-Z][a-z]+) (film|movie)', text)
            if match:
                country = match.group(1)

    # Remove duplicate names in director and country
    if director:
        director = ', '.join(dict.fromkeys(director.split(', ')))
    if country:
        country = ', '.join(dict.fromkeys(country.split(', ')))

    return director, country, production


# ---------- Main Parsing Logic ----------
response = safe_request(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find the main table containing highest-grossing films
main_table = soup.find('table', class_='wikitable')
rows = main_table.find_all('tr')

films_data = []

# Iterate over each row in the table (skip header)
for row in rows[1:]:
    cells = row.find_all(['td', 'th'])
    if len(cells) < 5:
        continue  # skip rows that are too short

    # Extract link to the individual film page
    link = cells[2].find('a')
    if not link:
        continue

    # Extract basic data
    title = clean_text(link.get_text())
    film_url = "https://en.wikipedia.org" + link.get('href')
    
    year_text = clean_text(cells[4].get_text())
    year = int(year_text) if year_text.isdigit() else None
    
    box_office = clean_money(cells[3].get_text())

    

    # Parse additional info from film page
    director, country, production = parse_film_page(film_url)

    

    # Append data to list
    films_data.append({
        'title': title,
        'release_year': year,
        'directors': director,
        'box_office': box_office,
        'country': country,
        'production_companies': production
    })

    # Random delay to avoid hammering Wikipedia servers
    time.sleep(random.uniform(0.5, 1.2))


# ---------- Save to CSV ----------
df = pd.DataFrame(films_data)
df.to_csv('films.csv', index=False, encoding='utf-8-sig')

#print("\nDone!")

# Loading Data into SQLite Database

## Why SQLite?
SQLite provides structured data storage and convenient querying capabilities.

## Table Structure: `films`

| Column | Data Type | Description |
|--------|-----------|-------------|
| id | INTEGER PRIMARY KEY | Auto-increment unique identifier |
| title | TEXT NOT NULL | Film title |
| release_year | INTEGER | Year of release |
| director | TEXT | Director name(s) |
| box_office | REAL | Box office earnings in USD |
| country | TEXT | Country of origin |
| production_companies | TEXT | Production company/studio |

## Process Flow
1. Read CSV file `films.csv` (try multiple encodings)
2. Create connection to `films.db` database
3. Create table if it doesn't exist
4. Insert all rows from DataFrame into table
5. Commit changes and close connection

## Encoding Handling
Attempts encodings in order: utf-8-sig → utf-8 → cp1251 → latin-1 → iso-8859-1

In [3]:


# Try different encodings to safely read the CSV file
encodings = ['utf-8-sig', 'utf-8', 'cp1251', 'latin-1', 'iso-8859-1']

df = None
for enc in encodings:
    try:
        df = pd.read_csv('films.csv', encoding=enc)
        break
    except UnicodeDecodeError:
        continue

if df is None:
    print("Failed to read CSV with any of the specified encodings")
else:
    # Connect to SQLite database (or create it if it doesn't exist)
    conn = sqlite3.connect('films.db')
    cursor = conn.cursor()
    
    # Create the 'films' table if it doesn't exist
    # I have added the additional attribute production_companies to the table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS films (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        release_year INTEGER,
        director TEXT,
        box_office REAL,
        country TEXT,
        production_companies TEXT
    )
    ''')
    
    # Insert data from DataFrame into the database
    for _, row in df.iterrows():
        cursor.execute('''
        INSERT INTO films (title, release_year, director, box_office, country, production_companies)
        VALUES (?, ?, ?, ?, ?, ?)
        ''', (
            row['title'],
            row['release_year'],
            row['directors'],
            row['box_office'],
            row['country'],
            row['production_companies']
        ))
    # Commit changes and close the connection
    conn.commit()
    conn.close()
    
    

# Exporting Data to JSON Format

## Purpose
Convert SQLite data to JSON for frontend visualization (Cinema Galaxy 3D timeline).

## Process Steps
1. Connect to `films.db` SQLite database
2. Query all films ordered by box office (descending)
3. Exclude the internal `id` column
4. Convert each row to a dictionary
5. Replace empty production info with "N/A"
6. Write to `films.json` file with UTF-8 encoding

## Output Format Example
```json
{
  "title": "Avatar",
  "release_year": 2009,
  "director": "James Cameron",
  "box_office": 2923710708,
  "country": "United States",
  "production_companies": "20th Century Fox; Lightstorm Entertainment"
}

In [4]:

# Connect to the SQLite database (films.db)
conn = sqlite3.connect('films.db')
cursor = conn.cursor()

# Fetch all data from the 'films' table, excluding the 'id' column
# Sorting by box_office in descending order
cursor.execute("""
    SELECT title, release_year, director, box_office, country, production_companies 
    FROM films
    ORDER BY box_office DESC
""")

rows = cursor.fetchall()

# Convert the fetched rows into a list of dictionaries
films_json = []
for row in rows:
    films_json.append({
        "title": row[0],
        "release_year": row[1],
        "director": row[2],
        "box_office": row[3],
        "country": row[4],
        "production_companies": row[5] if row[5] else "N/A" # Replace empty production info with "N/A"
    })

# Export the data to a JSON file for frontend use
with open('films.json', 'w', encoding='utf-8') as f:
    json.dump(films_json, f, ensure_ascii=False, indent=2)
# Close the database connection
conn.close()